In [ ]:
import os

GIT = True
REPO_NAME = "generative_models_for_image"

if GIT:
    if not os.path.exists(REPO_NAME):
        !git clone https://github.com/Mayeul84/generative_models_for_image

import os
os.chdir(REPO_NAME)

In [ ]:
import torch
import matplotlib.pyplot as plt
import yaml
import argparse

from data import get_dataset, get_dataloader
from guided_diffusion.unet import create_model
from guided_diffusion.gaussian_diffusion import create_sampler
from util.img_utils import clear_color
from tasks import create_operator
from gibbs_sampler import GibbsSampler
import numpy as np
from PIL import Image

from utils_image import *

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def load_yaml(file_path: str) -> dict:
    with open(file_path) as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    return config

!wget -nc -O data/ffhq256-1k-validation.zip 'https://www.dropbox.com/scl/fi/pppstbdsf0em6o0qscruc/ffhq256-1k-validation.zip?rlkey=xl7nwv2nxb6yvsirr3wad77hm'
!unzip -nq data/ffhq256-1k-validation.zip

!wget -nc -O models/ffhq_10m.pt 'https://www.dropbox.com/scl/fi/pq72vxzxcbygieq5z4gvf/ffhq_10m.pt?rlkey=5sxdj6r4o9f7b7bbp5fxg2f5r'

In [ ]:
def load_yaml(file_path: str) -> dict:
    with open(file_path) as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    return config

args = {
    'data_config': 'configs/data_config.yaml',
    'model_config': 'configs/model_config.yaml',
    'diffusion_config': 'configs/diffusion_config.yaml',
    'operator_config': 'configs/operator_config.yaml',
    'gibbs_config': 'configs/gibbs_config.yaml',
    'gpu': 0,
    'save_dir': './results'
}

data_config = load_yaml(args['data_config'])
model_config = load_yaml(args['model_config'])
diffusion_config = load_yaml(args['diffusion_config'])
operator_config = load_yaml(args['operator_config'])
gibbs_config = load_yaml(args['gibbs_config'])

In [ ]:
# Load model
print("Load model")
model = create_model(**model_config)
model = model.to(device)
model.eval()

# Load diffusion sampler
print("Load sampler")
sampler = create_sampler(**diffusion_config)

# Load operator
print("Load operator")
operator = create_operator(**operator_config, device=device)

# Load dataset
print("Load dataset")
get_dataset("ffhq", root="data/ffhq256-1k-validation")
dataset = get_dataset("ffhq", root="data/ffhq256-1k-validation")
num_test_images = len(dataset)
dataloader = get_dataloader(dataset, batch_size=1, num_workers=0, train=False)

In [ ]:
### Choose the image
idx = 462
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'),device=device)
imgshape = x0.shape

In [ ]:
def tensor2im(x):
    x = 0.5+0.5*x # [-1,1]->[0,1]
    return x.detach().cpu().permute(2,3,1,0).squeeze()

def im2tensor(x):
    x = torch.tensor(x,device=device)
    x = 2*x-1 # [0,1]->[-1,1]
    return x.permute(2,0,1).unsqueeze(0)

def rgb2gray(u):
    return 0.2989 * u[:,:,0] + 0.5870 * u[:,:,1] + 0.1140 * u[:,:,2]

def str2(chars):
    return "{:.2f}".format(chars)

def psnr(uref,ut,M=2):
    rmse = np.sqrt(np.mean((np.array(uref.cpu())-np.array(ut.cpu()))**2))
    return 20*np.log10(M/rmse)

# viewimage
import tempfile
import IPython

def viewimage(im, normalize=True,vmin=-1,vmax=1,z=1,order=0,titre='',displayfilename=False):
    im = im.detach().cpu().permute(2,3,1,0).squeeze()
    imin= np.array(im).astype(np.float32)
    channel_axis = 2 if len(im.shape)>2 else None
    if z!=1:
        from skimage.transform import rescale
        imin = rescale(imin, z, order=order, channel_axis=channel_axis)
    if normalize:
        if vmin is None:
            vmin = imin.min()
        if vmax is None:
            vmax = imin.max()
        if np.abs(vmax-vmin)>1e-10:
            imin = (imin.clip(vmin,vmax)-vmin)/(vmax-vmin)
        else:
            imin = vmin
    else:
        imin=imin.clip(0,255)/255
    imin=(imin*255).astype(np.uint8)
    filename=tempfile.mktemp(titre+'.png')
    if displayfilename:
        print (filename)
    plt.imsave(filename, imin, cmap='gray')
    IPython.display.display(IPython.display.Image(filename))

idx = np.random.randint(1000)
print('Image', str(idx).zfill(5))

# Test display function:
img = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
print(img.min().item(),img.max().item())
viewimage(img)

#set BSNR
BSNR = 60 # SNR expressed in decibels
P_signal = img.var() # signal power
sigma = np.sqrt((P_signal/10**(BSNR/10))) # standard deviation of the noise


In [ ]:
X = img.to(device)
sigma = sigma.to(device)
Y = operator.forward(X) + sigma*torch.randn(X.shape).to(device)
gibbs = GibbsSampler(Y=Y, 
                        sigma=sigma, 
                        operator=operator, 
                        sampler=sampler, 
                        model=model, 
                        device=device, 
                        **gibbs_config)

X_MC, Z_MC = gibbs.run()

fig, axes = plt.subplots(1, 4, figsize=(20, 20))

axes[0].imshow(clear_color(X))
axes[0].set_title('True image')
axes[0].axis('off');

axes[1].imshow(clear_color(Y))
axes[1].set_title('Noisy image')
axes[1].axis('off');

axes[2].imshow(clear_color(torch.mean(Z_MC[:,:,:,gibbs_config["N_bi"]:gibbs_config["N_MC"]], axis=-1)));
axes[2].set_title('Z Reconstructed image')
axes[2].axis('off');

axes[3].imshow(clear_color(torch.mean(X_MC[:,:,:,gibbs_config["N_bi"]:gibbs_config["N_MC"]], axis=-1)));
axes[3].set_title('X Reconstructed image')
axes[3].axis('off');

#plt.savefig(f"results/test.png", dpi=200, bbox_inches='tight')